In [0]:
%pip install "confluent-kafka[avro,schemaregistry]"
%pip install authlib
%restart_python

# Productor de transacciones bancarias

Este notebook publica transacciones financieras sinteticas en Avro. El esquema se registra automaticamente en Aiven Schema Registry con el subject `<topic>-value`.

Antes de ejecutar el productor, crea en Aiven el topic `banco.transacciones_tarjeta` o cambia `KAFKA_TOPIC` por el nombre que prefieras.

In [0]:
import random
import time
import uuid
from datetime import datetime, timezone

from confluent_kafka import Producer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext

## Parametros y esquema Avro

In [0]:
# ============================================================
# CONFIGURACION: modificar estos valores antes de ejecutar
# ============================================================
KAFKA_BOOTSTRAP_SERVERS = "kafka-utec-pe-01-utec-data.b.aivencloud.com:28780"
KAFKA_SECURITY_PROTOCOL = "SASL_SSL"
KAFKA_SASL_MECHANISMS = "SCRAM-SHA-256"
KAFKA_SASL_USERNAME = "avnadmin"
KAFKA_SASL_PASSWORD = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="KAFKA_SASL_PASSWORD")

# Certificado CA de Aiven en formato PEM.
CA_CERTIFICATE = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="KAFKA_CA_CERTIFICATE_PEM")

SCHEMA_REGISTRY_URL = "https://kafka-utec-pe-01-utec-data.b.aivencloud.com:28772"
SCHEMA_REGISTRY_USER = "avnadmin"
SCHEMA_REGISTRY_PWD = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="SCHEMA_REGISTRY_PASSWORD")

# Topic nuevo para eventos de actividad financiera.
KAFKA_TOPIC = "banco.transacciones_tarjeta"
EVENT_INTERVAL_SECONDS = 3.0

SCHEMA_AVRO = """
{
  "type": "record",
  "name": "TransaccionBancaria",
  "namespace": "banco.transacciones",
  "fields": [
    {"name": "transaccion_id", "type": "string"},
    {"name": "cliente_id", "type": "long"},
    {"name": "cuenta_id", "type": "long"},
    {"name": "tarjeta_id", "type": "string"},
    {"name": "tipo_transaccion", "type": "string"},
    {"name": "estado", "type": "string"},
    {"name": "monto", "type": "double"},
    {"name": "moneda", "type": "string"},
    {"name": "categoria_comercio", "type": "string"},
    {"name": "canal", "type": "string"},
    {"name": "pais", "type": "string"},
    {"name": "fecha_operacion", "type": {"type": "int", "logicalType": "date"}},
    {"name": "timestamp_evento", "type": {"type": "long", "logicalType": "timestamp-millis"}}
  ]
}
"""

# Datos sinteticos; no representan clientes reales.
CUENTAS = {
    2001001: {"cliente_id": 501001, "tarjeta_id": "CARD-2001001"},
    2001002: {"cliente_id": 501002, "tarjeta_id": "CARD-2001002"},
    2001003: {"cliente_id": 501003, "tarjeta_id": "CARD-2001003"},
    2001004: {"cliente_id": 501004, "tarjeta_id": "CARD-2001004"},
    2001005: {"cliente_id": 501005, "tarjeta_id": "CARD-2001005"}
}
TIPOS_TRANSACCION = ["COMPRA", "RETIRO", "TRANSFERENCIA", "PAGO_SERVICIO"]
ESTADOS = ["APROBADA", "APROBADA", "APROBADA", "RECHAZADA"]
MONEDAS = ["PEN", "USD"]
CATEGORIAS_COMERCIO = ["RETAIL", "RESTAURANTE", "SUPERMERCADO", "SALUD", "EDUCACION", "SERVICIOS"]
CANALES = ["POS", "APP", "WEB", "ATM"]
PAISES = ["PE", "CO", "CL", "MX"]

In [0]:
def main():
    """Conecta con Kafka y publica transacciones bancarias aleatorias."""
    producer = Producer({
        "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
        "security.protocol": KAFKA_SECURITY_PROTOCOL,
        "sasl.mechanisms": KAFKA_SASL_MECHANISMS,
        "sasl.username": KAFKA_SASL_USERNAME,
        "sasl.password": KAFKA_SASL_PASSWORD,
        "ssl.ca.pem": CA_CERTIFICATE,
        "client.id": "banco-transacciones-random-producer",
    })

    schema_registry = SchemaRegistryClient({
        "url": SCHEMA_REGISTRY_URL,
        "basic.auth.user.info": f"{SCHEMA_REGISTRY_USER}:{SCHEMA_REGISTRY_PWD}",
    })
    serializer = AvroSerializer(schema_registry, SCHEMA_AVRO)

    print(f"Productor iniciado en {KAFKA_TOPIC}. Presiona Ctrl+C para detenerlo.")
    contador = 0

    try:
        while True:
            cuenta_id = random.choice(list(CUENTAS))
            cuenta = CUENTAS[cuenta_id]
            ahora = datetime.now(timezone.utc)
            transaccion_id = f"TXN-{uuid.uuid4().hex[:16].upper()}"
            monto = round(random.uniform(10.00, 3500.00), 2)

            evento = {
                "transaccion_id": transaccion_id,
                "cliente_id": cuenta["cliente_id"],
                "cuenta_id": cuenta_id,
                "tarjeta_id": cuenta["tarjeta_id"],
                "tipo_transaccion": random.choice(TIPOS_TRANSACCION),
                "estado": random.choice(ESTADOS),
                "monto": monto,
                "moneda": random.choice(MONEDAS),
                "categoria_comercio": random.choice(CATEGORIAS_COMERCIO),
                "canal": random.choice(CANALES),
                "pais": random.choice(PAISES),
                "fecha_operacion": ahora.date(),
                "timestamp_evento": int(ahora.timestamp() * 1000),
            }

            producer.produce(
                topic=KAFKA_TOPIC,
                key=str(cuenta_id),
                value=serializer(
                    evento,
                    SerializationContext(KAFKA_TOPIC, MessageField.VALUE),
                ),
            )
            producer.poll(0)
            contador += 1
            print(
                f"Evento #{contador}: transaccion_id={transaccion_id}, "
                f"cuenta={cuenta_id}, tipo={evento['tipo_transaccion']}, "
                f"estado={evento['estado']}, monto={monto:.2f} {evento['moneda']}"
            )
            time.sleep(EVENT_INTERVAL_SECONDS)
    except KeyboardInterrupt:
        print(f"\nProductor detenido. Eventos generados: {contador}")
    finally:
        producer.flush(10)

main()